# 23-13 · Перемещаем файлы во временном каталоге

Практика к разделу [«Безопасно перемещаем файлы»](../../site/chapters/glava-23/23-13-peremeshaem-fajly.html). Использует настоящий пакет `safesort`.

## Цель

Вызвать настоящую `safesort.executor.apply_plan()` во временном каталоге и убедиться, что она перемещает файлы и отказывается перезаписывать уже занятое место назначения.

## Рабочий пример

In [1]:
import tempfile
from pathlib import Path

from safesort.executor import apply_plan
from safesort.models import MoveOperation, SortPlan

tmpdir = tempfile.TemporaryDirectory()
koren = Path(tmpdir.name)

istochnik = koren / "otchet.pdf"
istochnik.write_text("содержимое отчёта", encoding="utf-8")

naznachenie = koren / "Sorted" / "documents" / "otchet.pdf"
plan = SortPlan(root=koren, operations=(MoveOperation(source=istochnik, destination=naznachenie),))

rezultaty = apply_plan(plan)
print(rezultaty)

[CompletedMove(source=PosixPath('/tmp/tmp6q6839p8/otchet.pdf'), destination=PosixPath('/tmp/tmp6q6839p8/Sorted/documents/otchet.pdf'), completed=True, error=None)]


## Проверка результата

In [2]:
assert rezultaty[0].completed is True
assert not istochnik.exists()
assert naznachenie.exists()
assert naznachenie.read_text(encoding="utf-8") == "содержимое отчёта"
print("Верно: файл перемещён, содержимое не повреждено, исходное место пусто.")

Верно: файл перемещён, содержимое не повреждено, исходное место пусто.


## Эксперимент — существующий файл в месте назначения не перезаписывается

In [3]:
istochnik2 = koren / "zametka.txt"
istochnik2.write_text("новый текст", encoding="utf-8")

naznachenie2 = koren / "Sorted" / "documents" / "zametka.txt"
naznachenie2.parent.mkdir(parents=True, exist_ok=True)
naznachenie2.write_text("уже лежавший здесь текст", encoding="utf-8")

plan2 = SortPlan(root=koren, operations=(MoveOperation(source=istochnik2, destination=naznachenie2),))
rezultaty2 = apply_plan(plan2)

assert rezultaty2[0].completed is False
assert "already exists" in rezultaty2[0].error
assert istochnik2.exists()
assert naznachenie2.read_text(encoding="utf-8") == "уже лежавший здесь текст"
print("Верно: apply_plan отказался перезаписать существующий файл.")

Refusing to overwrite existing destination: /tmp/tmp6q6839p8/Sorted/documents/zametka.txt


Верно: apply_plan отказался перезаписать существующий файл.


## Задание ★★ Самостоятельная задача

Переместите два файла в одном плане и проверьте, что оба оказались на месте.

In [4]:
a = koren / "a.txt"
b = koren / "b.txt"
a.write_text("A", encoding="utf-8")
b.write_text("B", encoding="utf-8")

plan3 = SortPlan(
    root=koren,
    operations=(
        MoveOperation(source=a, destination=koren / "Sorted" / "documents" / "a.txt"),
        MoveOperation(source=b, destination=koren / "Sorted" / "documents" / "b.txt"),
    ),
)
rezultaty3 = apply_plan(plan3)

assert all(r.completed for r in rezultaty3)
assert (koren / "Sorted" / "documents" / "a.txt").exists()
assert (koren / "Sorted" / "documents" / "b.txt").exists()
print("Верно: оба файла перемещены за один apply_plan.")

tmpdir.cleanup()

Верно: оба файла перемещены за один apply_plan.
